> ### ▶ Running this notebook in Google Colab (recommended)> 1. Go to [colab.research.google.com](https://colab.research.google.com)> 2. Click **File → Upload notebook**> 3. Select this `.ipynb` file from your downloaded course ZIP> 4. Click **Runtime → Run all**>> Alternatively: upload the whole ZIP to **Google Drive**, then double-click any notebook — it opens in Colab automatically.>> _No installation required. All you need is a free Google account._

## 5.1 — Setup

In [ ]:
import sysIN_COLAB = 'google.colab' in sys.modulesif IN_COLAB:    import subprocess    subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', '--quiet'])import pandas as pdimport numpy as npprint("✅ Ready")

---## 5.2 — Model Structure: The Three-Layer RuleEvery financial model should have three clearly separated layers:| Layer | What it contains | In Python ||---|---|---|| **Inputs** | Assumptions, parameters, raw data | A dict or config file at the top || **Engine** | Calculations and logic | Functions that take inputs and return outputs || **Outputs** | Tables, charts, exports | Calls to the engine, formatting, file writes |This mirrors how good Excel models are structured (input tab → calculation tab → output tab) but enforces the separation in code.

In [ ]:
# ── LAYER 1: INPUTS ──────────────────────────────────────────────────────────# All assumptions in one place. Well-named. Commented.model_inputs = {    # Company    'company_name':     'Example Corp',    'reporting_ccy':    'GBP',    'base_year':        2024,    # Revenue model    'base_revenue':     15_000,       # £000s    'growth_rates':     [0.10, 0.09, 0.08],  # Y1, Y2, Y3    # Margin structure    'gross_margin':     0.60,    'opex_pct':         0.27,    'da_fixed':         600,          # £000s, not % (it's a fixed charge)    # Capital structure    'tax_rate':         0.25,}print("Inputs defined. Keys:", list(model_inputs.keys()))

In [ ]:
# ── LAYER 2: ENGINE ──────────────────────────────────────────────────────────# Pure functions: inputs in, outputs out. No side effects, no globals.def project_revenue(base, growth_rates):    """Project revenue forward given a base and list of annual growth rates."""    revenues = [base]    for g in growth_rates:        revenues.append(revenues[-1] * (1 + g))    return revenues[1:]  # Exclude base yeardef build_income_statement(inputs):    """    Build a multi-year P&L from an inputs dictionary.    Returns a DataFrame indexed by year.    """    i = inputs    years = [f"{i['base_year'] + y}" for y in range(1, len(i['growth_rates']) + 1)]    revenues     = project_revenue(i['base_revenue'], i['growth_rates'])    gross_profit = [r * i['gross_margin'] for r in revenues]    opex         = [r * i['opex_pct'] for r in revenues]    ebitda       = [gp - op for gp, op in zip(gross_profit, opex)]    ebit         = [e - i['da_fixed'] for e in ebitda]    nopat        = [e * (1 - i['tax_rate']) for e in ebit]    return pd.DataFrame({        'Revenue':       revenues,        'Gross Profit':  gross_profit,        'EBITDA':        ebitda,        'EBIT':          ebit,        'NOPAT':         nopat,    }, index=years).round(1)# ── LAYER 3: OUTPUTS ─────────────────────────────────────────────────────────pl = build_income_statement(model_inputs)print(f"P&L for {model_inputs['company_name']} ({model_inputs['reporting_ccy']}000s)")pl

---## 5.3 — Model ValidationGood models check their own arithmetic. In Excel this is manual — someone eyeballs the numbers. In Python, you can write assertions that raise an error if the model is internally inconsistent.

In [ ]:
def validate_model(pl, inputs):    """    Run sanity checks on the model output.    Raises AssertionError if any check fails.    """    errors = []    # Check 1: EBITDA should always be positive (for this business)    if (pl['EBITDA'] <= 0).any():        errors.append("❌ EBITDA is negative in at least one year")    # Check 2: Gross profit should be between 0% and 100% of revenue    gm = pl['Gross Profit'] / pl['Revenue']    if not ((gm > 0) & (gm < 1)).all():        errors.append("❌ Gross margin is outside 0-100% range")    # Check 3: Revenue should grow year-on-year    if not (pl['Revenue'].diff().dropna() > 0).all():        errors.append("❌ Revenue is not growing in all years")    # Check 4: NOPAT should be less than EBIT (tax reduces it)    if not (pl['NOPAT'] < pl['EBIT']).all():        errors.append("❌ NOPAT >= EBIT — tax calculation looks wrong")    if errors:        for e in errors:            print(e)        raise ValueError("Model validation failed. Fix the errors above before using outputs.")    else:        print("✅ All validation checks passed.")validate_model(pl, model_inputs)

---## 5.4 — Git Version Control (No More Filename Suffixes)> "model_FINAL_v3_REAL_used_this_one.xlsx"Sound familiar? Git solves this permanently. Here's the minimum you need to know.### One-time setup```bash# In your terminal, inside your project foldergit initgit add .git commit -m "Initial model: 3-year P&L for Example Corp"```### Every time you make a meaningful change```bashgit add .git commit -m "Updated growth assumptions after board review"```### See the full history```bashgit log --oneline```### Go back to a previous version```bashgit checkout <commit-hash>```### Recommended folder structure for a model project```example-corp-model/├── README.md               ← What is this model, who built it, last updated├── inputs/│   └── assumptions.py      ← Your inputs dict (version-controlled)├── data/│   └── raw_financials.xlsx ← Source data (never modify this directly)├── model/│   └── engine.py           ← Your functions (build_pl, validate, etc.)├── notebooks/│   └── analysis.ipynb      ← Your working notebook└── outputs/    └── .gitkeep            ← Git tracks the folder but not generated outputs```> **Note:** If you work at a bank, check your firm's policy on using personal GitHub accounts for work-related code. For personal projects and course work, a free GitHub account is fine.

---## 5.5 — Writing Reusable Model TemplatesThe real payoff of well-structured Python models is reusability. Once `build_income_statement()` works for one company, it works for any company — just pass in different inputs.

In [ ]:
# Run the same model engine for three different companiescompanies = {    'Alpha Ltd': {**model_inputs, 'company_name': 'Alpha Ltd',                  'base_revenue': 8_000, 'growth_rates': [0.15, 0.12, 0.10],                  'gross_margin': 0.65},    'Beta plc':  {**model_inputs, 'company_name': 'Beta plc',                  'base_revenue': 25_000, 'growth_rates': [0.06, 0.05, 0.05],                  'gross_margin': 0.48},    'Gamma Co':  {**model_inputs, 'company_name': 'Gamma Co',                  'base_revenue': 5_000, 'growth_rates': [0.20, 0.18, 0.15],                  'gross_margin': 0.72},}# Build all three and extract Year 3 EBIT for comparisoncomparison = {}for name, inp in companies.items():    pl_c = build_income_statement(inp)    comparison[name] = {        'Y3 Revenue':      pl_c['Revenue'].iloc[-1],        'Y3 EBIT':         pl_c['EBIT'].iloc[-1],        'Y3 EBIT Margin':  f"{pl_c['EBIT'].iloc[-1]/pl_c['Revenue'].iloc[-1]*100:.1f}%"    }pd.DataFrame(comparison).T

---## 5.6 — Inline DocumentationA model your colleagues can't read is a model they can't trust. Good docstrings and comments are non-negotiable in professional work.

In [ ]:
# Example of a well-documented model functiondef calculate_wacc(    cost_of_equity: float,    cost_of_debt: float,    equity_weight: float,    debt_weight: float,    tax_rate: float,) -> float:    """    Calculate Weighted Average Cost of Capital (WACC).    Uses the standard after-tax formula:        WACC = E/V * Re + D/V * Rd * (1 - Tc)    Args:        cost_of_equity: Required return on equity (e.g. 0.10 for 10%)        cost_of_debt:   Pre-tax cost of debt (e.g. 0.045 for 4.5%)        equity_weight:  Equity as proportion of total capital (E/V)        debt_weight:    Debt as proportion of total capital (D/V)        tax_rate:       Corporate tax rate (e.g. 0.25 for 25%)    Returns:        WACC as a float (e.g. 0.0865 for 8.65%)    Raises:        ValueError: If weights do not sum to 1.0 (within tolerance)    Example:        >>> calculate_wacc(0.10, 0.045, 0.70, 0.30, 0.25)        0.08012...    """    if abs(equity_weight + debt_weight - 1.0) > 0.001:        raise ValueError(f"Weights must sum to 1.0, got {equity_weight + debt_weight}")    return (equity_weight * cost_of_equity) + (debt_weight * cost_of_debt * (1 - tax_rate))# Test itwacc = calculate_wacc(0.10, 0.045, 0.70, 0.30, 0.25)print(f"WACC: {wacc*100:.2f}%")

---## 5.7 — Key Takeaways- **Three-layer structure** (Inputs / Engine / Outputs) is the most important discipline in Python modelling — it makes models readable, testable, and reusable- **Validation functions** catch errors automatically — don't rely on eyeballing outputs- **Git** eliminates filename chaos and gives you a full audit trail — one commit message is worth a thousand "v3_FINAL" suffixes- **Well-typed, well-documented functions** are the difference between a model that lives for a week and one that gets reused for years- **Reusability** is the compounding return on doing this properly — one good engine function serves every company you ever model---## Up Next: Module 6 — Automation & ReportingWe'll build an automated monthly reporting pipeline: read updated data, run the model, and output a formatted Excel report — all with one command.